**Data Simulation**

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

# --- 1. Products Table ---
num_products = 100
product_ids = [f'P{i:03d}' for i in range(1, num_products + 1)]
product_categories = np.random.choice(['Electronics', 'Perishable Food', 'Non-Perishable Food', 'Apparel', 'Home Goods'], num_products)

df_products = pd.DataFrame({
    'product_id': product_ids,
    'product_name': [f'Product {i}' for i in range(1, num_products + 1)],
    'product_category': product_categories,
    'unit_of_measure': np.random.choice(['units', 'kg', 'liters'], num_products),
    'standard_material_cost_per_unit': np.round(np.random.uniform(5, 100, num_products), 2),
    'standard_labor_cost_per_unit': np.round(np.random.uniform(1, 20, num_products), 2),
    'shelf_life_days': np.where(product_categories == 'Perishable Food', np.random.randint(7, 90, num_products), np.random.randint(180, 730, num_products))
})
# Add some products with high standard costs to simulate high-value items
df_products.loc[np.random.choice(df_products.index, 10), 'standard_material_cost_per_unit'] = np.round(np.random.uniform(150, 500, 10), 2)


# --- 2. Sales Orders Table ---
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2024-12-31')
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

sales_data = []
for _ in range(20000): # Simulate 20,000 sales transactions
    order_date = pd.Timestamp(np.random.choice(date_range)) # Convert numpy datetime64 to pandas Timestamp
    product_id = np.random.choice(df_products['product_id'])
    quantity_sold = np.random.randint(1, 20)

    # Simulate seasonality (e.g., higher sales in Q4)
    if order_date.month in [10, 11, 12]:
        quantity_sold = int(quantity_sold * np.random.uniform(1.2, 2.0)) # Boost sales in Q4

    unit_selling_price = df_products[df_products['product_id'] == product_id]['standard_material_cost_per_unit'].iloc[0] * np.random.uniform(1.5, 3.0)
    sales_data.append({
        'order_id': f'SO{_ + 1:05d}',
        'order_date': order_date,
        'product_id': product_id,
        'quantity_sold': quantity_sold,
        'unit_selling_price': np.round(unit_selling_price, 2)
    })
df_sales = pd.DataFrame(sales_data)


# --- 3. Production Records Table ---
production_data = []
for _ in range(10000): # Simulate 10,000 production runs
    production_date = np.random.choice(date_range)
    product_id = np.random.choice(df_products['product_id'])
    quantity_produced = np.random.randint(50, 500)
    direct_labor_hours = np.round(quantity_produced * np.random.uniform(0.1, 0.5), 2)

    # Simulate some waste
    waste_quantity = 0
    waste_reason = 'None'
    if np.random.rand() < 0.15: # 15% chance of waste in a batch
        waste_quantity = np.random.randint(1, int(quantity_produced * 0.1)) # Up to 10% of batch
        waste_reason = np.random.choice(['Defective Material', 'Production Error', 'Spoilage', 'Overproduction'])

    production_data.append({
        'production_run_id': f'PR{_ + 1:05d}',
        'production_date': production_date,
        'product_id': product_id,
        'quantity_produced': quantity_produced,
        'direct_labor_hours': direct_labor_hours,
        'waste_quantity_produced': waste_quantity,
        'waste_reason': waste_reason
    })
df_production = pd.DataFrame(production_data)


# --- 4. Inventory Movements Table ---
# This is more complex and often derived from sales and production.
# For simplicity, we can simulate receipts (from production) and shipments (from sales).
# We will need to calculate daily inventory levels from these movements.

inventory_movements_data = []
# Add initial inventory
for product_id in df_products['product_id']:
    inventory_movements_data.append({
        'movement_id': f'INV000_INIT_{product_id}',
        'product_id': product_id,
        'movement_date': start_date - timedelta(days=1), # Day before start
        'movement_type': 'Initial Stock',
        'quantity_moved': np.random.randint(100, 500),
        'location_id': np.random.choice(['WH_A', 'WH_B'])
    })

# Add movements from production
for idx, row in df_production.iterrows():
    inventory_movements_data.append({
        'movement_id': f'INV_PR{row["production_run_id"]}',
        'product_id': row['product_id'],
        'movement_date': row['production_date'],
        'movement_type': 'Receipt (Production)',
        'quantity_moved': row['quantity_produced'],
        'location_id': np.random.choice(['WH_A', 'WH_B']) # Assume produced into a warehouse
    })

# Add movements from sales
for idx, row in df_sales.iterrows():
    inventory_movements_data.append({
        'movement_id': f'INV_SO{row["order_id"]}',
        'product_id': row['product_id'],
        'movement_date': row['order_date'], # Or delivery_date if simulated
        'movement_type': 'Shipment (Sale)',
        'quantity_moved': -row['quantity_sold'], # Negative for outgoing
        'location_id': np.random.choice(['WH_A', 'WH_B']) # Assume shipped from a warehouse
    })

df_inventory_movements = pd.DataFrame(inventory_movements_data).sort_values(by=['movement_date', 'product_id'])

# Save to CSVs
df_products.to_csv('products.csv', index=False)
df_sales.to_csv('sales_orders.csv', index=False)
df_production.to_csv('production_records.csv', index=False)
df_inventory_movements.to_csv('inventory_movements.csv', index=False)

print("Simulated data saved to CSV files.")

# **Step 1: Load the Data**

First, ensure our Jupyter Notebook is in the same directory as your CSV files, or provide the full path to them.

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

# Load the simulated datasets
try:
    df_products = pd.read_csv('products.csv')
    df_sales = pd.read_csv('sales_orders.csv')
    df_production = pd.read_csv('production_records.csv')
    df_inventory_movements = pd.read_csv('inventory_movements.csv')
    print("All CSV files loaded successfully!")
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Make sure the CSV files are in the same directory as your notebook, or provide the full path.")
    # We might want to exit or handle this error more gracefully in a real application
    exit()

# Display the first few rows of each DataFrame to confirm loading
print("\n--- df_products Head ---")
print(df_products.head())
print("\n--- df_sales Head ---")
print(df_sales.head())
print("\n--- df_production Head ---")
print(df_production.head())
print("\n--- df_inventory_movements Head ---")
print(df_inventory_movements.head())

# **Step 2: Initial Data Inspection & Cleaning**

It's crucial to check data types, missing values, and get a feel for the data's structure.

In [ ]:
# --- Initial Inspection ---
print("\n--- df_products Info ---")
df_products.info()
print("\n--- df_sales Info ---")
df_sales.info()
print("\n--- df_production Info ---")
df_production.info()
print("\n--- df_inventory_movements Info ---")
df_inventory_movements.info()

# --- Data Type Conversion ---
# Convert date columns to datetime objects
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])
df_production['production_date'] = pd.to_datetime(df_production['production_date'])
df_inventory_movements['movement_date'] = pd.to_datetime(df_inventory_movements['movement_date'])

# Ensure numerical columns are correctly typed (Pandas usually infers well, but good to check)
# For instance, ensure 'quantity_sold', 'quantity_produced', 'waste_quantity_produced' are integers/floats

print("\n--- Data types after conversion ---")
print(df_sales.info())
print(df_production.info())
print(df_inventory_movements.info())

# --- Check for Missing Values ---
print("\n--- Missing values in df_products ---")
print(df_products.isnull().sum())
print("\n--- Missing values in df_sales ---")
print(df_sales.isnull().sum())
print("\n--- Missing values in df_production ---")
print(df_production.isnull().sum())
print("\n--- Missing values in df_inventory_movements ---")
print(df_inventory_movements.isnull().sum())

# For this simulated data, there should be very few or no missing values,
# but in real-world data, this is where we would handle them (e.g., df.dropna(), df.fillna())

# --- Handle potential outliers or inconsistencies (if any were intentionally simulated) ---
# For this project, we'll assume the simulated data is relatively clean.
# In a real scenario, we might look at distributions:
# import matplotlib.pyplot as plt
# df_sales['quantity_sold'].hist()
# plt.title('Distribution of Quantity Sold')
# plt.show()

# **Step 3: Data Integration (Merging DataFrames)**

To calculate COGS and Waste metrics, we will need product cost information alongside sales and production records.

In [ ]:
# Merge sales data with product data to get cost information for each sold item
df_sales_merged = pd.merge(
    df_sales,
    df_products[['product_id', 'standard_material_cost_per_unit', 'standard_labor_cost_per_unit']],
    on='product_id',
    how='left'
)

# Merge production data with product data to get cost information for waste calculation
df_production_merged = pd.merge(
    df_production,
    df_products[['product_id', 'standard_material_cost_per_unit', 'standard_labor_cost_per_unit']],
    on='product_id',
    how='left'
)

print("\n--- df_sales_merged Head (with cost info) ---")
print(df_sales_merged.head())
print("\n--- df_production_merged Head (with cost info) ---")
print(df_production_merged.head())

# **Step 4: Metric Calculation (COGS, Waste, DOI)**

This is the most involved step.

**A. Calculate COGS**

In [ ]:
# Calculate estimated COGS for each sales transaction
# Simplified COGS: (Standard Material Cost + Standard Labor Cost) * Quantity Sold
df_sales_merged['cogs_per_unit'] = df_sales_merged['standard_material_cost_per_unit'] + df_sales_merged['standard_labor_cost_per_unit']
df_sales_merged['total_cogs'] = df_sales_merged['cogs_per_unit'] * df_sales_merged['quantity_sold']

# Aggregate COGS by date (e.g., monthly)
df_sales_merged['month_year'] = df_sales_merged['order_date'].dt.to_period('M')
monthly_cogs = df_sales_merged.groupby('month_year')['total_cogs'].sum().reset_index()
monthly_cogs['month_year'] = monthly_cogs['month_year'].dt.to_timestamp() # Convert back to datetime for plotting
monthly_cogs.rename(columns={'total_cogs': 'Monthly_COGS'}, inplace=True)

print("\n--- Monthly COGS ---")
print(monthly_cogs.head())

**B. Calculate Waste Metrics**

In [ ]:
# Calculate waste cost for each production run
df_production_merged['waste_cost_per_unit'] = df_production_merged['standard_material_cost_per_unit'] + df_production_merged['standard_labor_cost_per_unit']
df_production_merged['total_waste_cost'] = df_production_merged['waste_quantity_produced'] * df_production_merged['waste_cost_per_unit']

# Calculate waste rate per production run (if quantity_produced > 0)
df_production_merged['waste_rate'] = np.where(
    df_production_merged['quantity_produced'] > 0,
    (df_production_merged['waste_quantity_produced'] / df_production_merged['quantity_produced']) * 100,
    0 # If no quantity produced, waste rate is 0
)

# Aggregate Waste by date (e.g., monthly) and reason
df_production_merged['month_year'] = df_production_merged['production_date'].dt.to_period('M')
monthly_waste_cost = df_production_merged.groupby('month_year')['total_waste_cost'].sum().reset_index()
monthly_waste_cost['month_year'] = monthly_waste_cost['month_year'].dt.to_timestamp()
monthly_waste_cost.rename(columns={'total_waste_cost': 'Monthly_Waste_Cost'}, inplace=True)

waste_by_reason = df_production_merged.groupby('waste_reason')['total_waste_cost'].sum().sort_values(ascending=False).reset_index()

print("\n--- Monthly Waste Cost ---")
print(monthly_waste_cost.head())
print("\n--- Waste Cost by Reason ---")
print(waste_by_reason)

**C. Calculate Daily Inventory Levels (Crucial for DOI)**

This is the most complex part. We need to calculate the stock on hand for each product for every day.

In [ ]:
# Sort inventory movements to ensure correct cumulative sum
df_inventory_movements = df_inventory_movements.sort_values(by=['product_id', 'movement_date'])

# Calculate cumulative quantity for each product over time
df_inventory_movements['cumulative_quantity'] = df_inventory_movements.groupby('product_id')['quantity_moved'].cumsum()

# Create a full date range for the entire period
min_date = df_inventory_movements['movement_date'].min()
max_date = df_inventory_movements['movement_date'].max()
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')

# Create a DataFrame with all product-date combinations
all_product_dates = pd.MultiIndex.from_product([df_products['product_id'], full_date_range], names=['product_id', 'date']).to_frame(index=False)
all_product_dates['date'] = pd.to_datetime(all_product_dates['date']) # Ensure datetime type

# Merge cumulative quantities onto the full product-date grid
# This will bring the last known cumulative quantity for each product on each day
df_daily_inventory = pd.merge(
    all_product_dates,
    df_inventory_movements[['product_id', 'movement_date', 'cumulative_quantity']],
    left_on=['product_id', 'date'],
    right_on=['product_id', 'movement_date'],
    how='left'
)

# Sort and forward-fill the cumulative quantity for each product
df_daily_inventory = df_daily_inventory.sort_values(by=['product_id', 'date'])
df_daily_inventory['cumulative_quantity'] = df_daily_inventory.groupby('product_id')['cumulative_quantity'].ffill()

# Fill any remaining NaNs (e.g., for products that had no initial stock or movements) with 0
df_daily_inventory['cumulative_quantity'] = df_daily_inventory['cumulative_quantity'].fillna(0)

# Drop the redundant movement_date column
df_daily_inventory.drop(columns=['movement_date'], inplace=True)

# Merge with product costs to get inventory value
df_daily_inventory = pd.merge(
    df_daily_inventory,
    df_products[['product_id', 'standard_material_cost_per_unit', 'standard_labor_cost_per_unit']],
    on='product_id',
    how='left'
)
df_daily_inventory['unit_cost'] = df_daily_inventory['standard_material_cost_per_unit'] + df_daily_inventory['standard_labor_cost_per_unit']
df_daily_inventory['inventory_value'] = df_daily_inventory['cumulative_quantity'] * df_daily_inventory['unit_cost']

# Aggregate daily inventory value to monthly average inventory value
df_daily_inventory['month_year'] = df_daily_inventory['date'].dt.to_period('M')
monthly_avg_inventory_value = df_daily_inventory.groupby('month_year')['inventory_value'].mean().reset_index()
monthly_avg_inventory_value['month_year'] = monthly_avg_inventory_value['month_year'].dt.to_timestamp()
monthly_avg_inventory_value.rename(columns={'inventory_value': 'Monthly_Avg_Inventory_Value'}, inplace=True)

print("\n--- Monthly Average Inventory Value ---")
print(monthly_avg_inventory_value.head())

**D. Calculate Days of Inventory (DOI)**

Now that we have monthly COGS and monthly average inventory value, we can calculate DOI.

In [ ]:
# Merge monthly COGS and monthly average inventory value
df_metrics = pd.merge(
    monthly_cogs,
    monthly_avg_inventory_value,
    on='month_year',
    how='inner' # Use inner join to ensure we only have months with both COGS and inventory data
)

# Calculate DOI
# DOI = (Average Inventory Value / COGS) * Number of Days in Period
# For monthly DOI, Number of Days in Period can be approximated as 30.44 (avg days in month) or use actual days in month
df_metrics['days_in_month'] = df_metrics['month_year'].dt.daysinmonth # Get actual days in each month

df_metrics['DOI'] = (df_metrics['Monthly_Avg_Inventory_Value'] / df_metrics['Monthly_COGS']) * df_metrics['days_in_month']

# Handle cases where Monthly_COGS might be 0 to avoid division by zero
df_metrics['DOI'] = df_metrics['DOI'].replace([np.inf, -np.inf], np.nan) # Replace inf with NaN
df_metrics['DOI'] = df_metrics['DOI'].fillna(0) # Or another suitable value, e.g., a very high number if COGS is 0

print("\n--- Final Monthly Metrics (COGS, Avg Inventory Value, DOI) ---")
print(df_metrics.head())
print(df_metrics.tail())

# **Step 5: EDA and Root Cause Analysis**

This is where we will use our calculated metrics to uncover trends, identify potential issues, and gain insights into the simulated supply chain's performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a style for better aesthetics
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6) # Standard figure size

print("Starting Exploratory Data Analysis (EDA)...")

# --- 1. Time Series Trends: Monthly COGS, Waste Cost, and DOI ---

# Ensure the month_year column is in datetime format for plotting
df_metrics['month_year'] = pd.to_datetime(df_metrics['month_year'])
monthly_cogs['month_year'] = pd.to_datetime(monthly_cogs['month_year'])
monthly_waste_cost['month_year'] = pd.to_datetime(monthly_waste_cost['month_year'])


plt.figure(figsize=(15, 15))

# Plot Monthly COGS
plt.subplot(3, 1, 1) # 3 rows, 1 column, 1st plot
sns.lineplot(x='month_year', y='Monthly_COGS', data=monthly_cogs)
plt.title('Monthly Cost of Goods Sold (COGS) Trend')
plt.xlabel('Date')
plt.ylabel('Total COGS')
plt.grid(True)
plt.tight_layout()

# Plot Monthly Waste Cost
plt.subplot(3, 1, 2) # 3 rows, 1 column, 2nd plot
sns.lineplot(x='month_year', y='Monthly_Waste_Cost', data=monthly_waste_cost, color='orange')
plt.title('Monthly Waste Cost Trend')
plt.xlabel('Date')
plt.ylabel('Total Waste Cost')
plt.grid(True)
plt.tight_layout()

# Plot Monthly Days of Inventory (DOI)
plt.subplot(3, 1, 3) # 3 rows, 1 column, 3rd plot
sns.lineplot(x='month_year', y='DOI', data=df_metrics, color='green')
plt.title('Monthly Days of Inventory (DOI) Trend')
plt.xlabel('Date')
plt.ylabel('DOI (Days)')
plt.grid(True)
plt.tight_layout()

plt.suptitle('Overall Supply Chain Performance Trends (2023-2024)', y=1.02, fontsize=16)
plt.show()


# --- 2. Breakdowns by Product Category ---

# Merge product category info to sales and production data for aggregation
df_sales_category = pd.merge(df_sales_merged, df_products[['product_id', 'product_category']], on='product_id', how='left')
df_production_category = pd.merge(df_production_merged, df_products[['product_id', 'product_category']], on='product_id', how='left')
df_inventory_category = pd.merge(df_daily_inventory, df_products[['product_id', 'product_category']], on='product_id', how='left')


# COGS by Product Category
cogs_by_category = df_sales_category.groupby('product_category')['total_cogs'].sum().sort_values(ascending=False).reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(x='total_cogs', y='product_category', data=cogs_by_category, palette='viridis')
plt.title('Total COGS by Product Category')
plt.xlabel('Total COGS')
plt.ylabel('Product Category')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

# Waste Cost by Product Category
waste_cost_by_category = df_production_category.groupby('product_category')['total_waste_cost'].sum().sort_values(ascending=False).reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(x='total_waste_cost', y='product_category', data=waste_cost_by_category, palette='magma')
plt.title('Total Waste Cost by Product Category')
plt.xlabel('Total Waste Cost')
plt.ylabel('Product Category')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

# Average DOI by Product Category
# Calculate average DOI for each product, then average for category
product_doi = df_daily_inventory.groupby('product_id')['cumulative_quantity'].mean().reset_index()
product_doi = pd.merge(product_doi, df_products[['product_id', 'product_category', 'standard_material_cost_per_unit', 'standard_labor_cost_per_unit']], on='product_id', how='left')
product_doi['unit_cost'] = product_doi['standard_material_cost_per_unit'] + product_doi['standard_labor_cost_per_unit']
product_doi['avg_inventory_value'] = product_doi['cumulative_quantity'] * product_doi['unit_cost']

# To get a meaningful DOI per category, it's often (Total Avg Inventory Value / Total COGS) * Days
# For simplicity in visualization, let's look at average inventory value per category
avg_inventory_value_by_category = product_doi.groupby('product_category')['avg_inventory_value'].sum().sort_values(ascending=False).reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(x='avg_inventory_value', y='product_category', data=avg_inventory_value_by_category, palette='cividis')
plt.title('Average Inventory Value by Product Category (Proxy for DOI Contribution)')
plt.xlabel('Average Inventory Value')
plt.ylabel('Product Category')
plt.grid(axis='x')
plt.tight_layout()
plt.show()


# --- 3. Deeper Dive into Waste Analysis ---

# Waste Cost by Reason
plt.figure(figsize=(10, 7))
sns.barplot(x='total_waste_cost', y='waste_reason', data=waste_by_reason, palette='rocket')
plt.title('Total Waste Cost by Reason')
plt.xlabel('Total Waste Cost')
plt.ylabel('Waste Reason')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

# Waste Rate Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_production_merged['waste_rate'][df_production_merged['waste_rate'] > 0], bins=20, kde=True, color='purple')
plt.title('Distribution of Waste Rates (for batches with waste)')
plt.xlabel('Waste Rate (%)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


# --- 4. Root Cause Exploration / Correlations (Example: Production Volume vs. Waste) ---

# Aggregate production data by month to see if higher production correlates with higher waste
monthly_production_summary = df_production_merged.groupby('month_year').agg(
    total_quantity_produced=('quantity_produced', 'sum'),
    total_waste_quantity=('waste_quantity_produced', 'sum')
).reset_index()

# Calculate monthly waste rate based on total produced
monthly_production_summary['monthly_waste_rate'] = (
    monthly_production_summary['total_waste_quantity'] / monthly_production_summary['total_quantity_produced']
) * 100
monthly_production_summary['month_year'] = monthly_production_summary['month_year'].dt.to_timestamp()


plt.figure(figsize=(10, 6))
sns.scatterplot(x='total_quantity_produced', y='total_waste_quantity', data=monthly_production_summary, hue='month_year', size='monthly_waste_rate', sizes=(20, 400), palette='coolwarm')
plt.title('Monthly Production Volume vs. Total Waste Quantity')
plt.xlabel('Total Quantity Produced (Monthly)')
plt.ylabel('Total Waste Quantity (Monthly)')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nEDA complete. Visualizations provide initial insights into COGS, Waste, and DOI drivers.")

To export our Final DataFrames to CSV for Power BI

In [ ]:
# Save the prepared DataFrames to CSV for Power BI
df_products.to_csv('pb_products.csv', index=False)
df_sales_merged.to_csv('pb_sales_merged.csv', index=False)
df_production_merged.to_csv('pb_production_merged.csv', index=False)
df_metrics.to_csv('pb_monthly_metrics.csv', index=False) # This is crucial for monthly trends

# We might also want these for specific breakdowns
df_sales_category.to_csv('pb_sales_category.csv', index=False)
df_production_category.to_csv('pb_production_category.csv', index=False)
waste_by_reason.to_csv('pb_waste_by_reason.csv', index=False)

print("\nData exported to CSVs for Power BI.")